# RAG pipeline experiments — LlamaIndex + pgvector + Gemini

One stage per cell, with printed intermediates. Runtime: `backend/.venv`
(select it as the kernel). Stage order:

0. Setup
1. LOAD — pypdf pages
2. PARSE — `SectionNodeParser` (sections + sentence chunks)
3. EMBED — fastembed vectors
4. STORE — `PGVectorStore` (HNSW)
5. RETRIEVE — session-filtered retriever with scores
6. ANSWER — Gemini response synthesizer
7. PLAYGROUND — tweak knobs and watch retrieval change

In [ ]:
# 0. SETUP — path, env, key check
import os
import sys

sys.path.insert(0, "..")
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GEMINI_API_KEY"), "Set GEMINI_API_KEY in backend/.env"
print("setup ok")

In [ ]:
# 1. LOAD — pypdf gives one page per document; metadata carries source + page
from app.services.node_parser import load_pdf_pages

PDF_PATH = "/Users/saiteja/Downloads/SriVenkataSivaSaiTejaTankala_Resume.pdf"
pages = load_pdf_pages(PDF_PATH)

print(f"{len(pages)} pages")
print("page metadata:", pages[0].metadata)
print(pages[0].text[:300])

In [ ]:
# 2. PARSE — sections from heading heuristics, chunks = N whole sentences
from app.services.node_parser import SectionNodeParser

parser = SectionNodeParser(
    session_id="notebook", sentences_per_chunk=6, sentence_overlap=1
)
nodes = parser.get_nodes_from_documents(pages)

print(f"{len(nodes)} nodes")
print("metadata:", nodes[0].metadata)
print(nodes[0].get_content()[:300])

In [ ]:
# 3. EMBED — local ONNX vectors (downloads ~130MB on first use, then cached)
from app.services.embeddings import FastEmbedEmbedding

embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
vec = embed_model.get_text_embedding(nodes[0].get_content())

print(f"{len(vec)} dims; first 5: {vec[:5]}")

In [ ]:
# 4. STORE — pgvector table 'rag_nodes' with an HNSW index (cosine)
from sqlalchemy.engine import make_url
from llama_index.core import Settings, VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore

from app.core.config import settings as app_settings

Settings.embed_model = embed_model

url = make_url(app_settings.database_url)
store = PGVectorStore.from_params(
    database=url.database,
    host=url.host,
    port=url.port,
    user=url.username,
    password=url.password,
    table_name="rag_nodes",
    embed_dim=app_settings.embed_dim,
    hnsw_kwargs={
        "hnsw_m": app_settings.hnsw_m,
        "hnsw_ef_construction": app_settings.hnsw_ef_construction,
        "hnsw_ef_search": app_settings.hnsw_ef_search,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
from llama_index.core.storage.storage_context import StorageContext
index = VectorStoreIndex(
    nodes=nodes,
    storage_context=StorageContext.from_defaults(vector_store=store),
)
print("indexed into rag_nodes (HNSW)")

In [ ]:
# 5. RETRIEVE — metadata filter scopes results to this session's nodes
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters

retriever = index.as_retriever(
    similarity_top_k=10,
    filters=MetadataFilters(
        filters=[ExactMatchFilter(key="session_id", value="notebook")]
    ),
)
question = "What experience does the candidate have?"  # <- change me
hits = retriever.retrieve(question)

for h in hits[:5]:
    print(round(h.score, 3), f"p.{h.node.metadata['page']}", h.node.metadata["section"])

In [ ]:
# 6. ANSWER — Gemini synthesizes from exactly the retrieved nodes
from llama_index.core.prompts import PromptTemplate
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.llms.google_genai import GoogleGenAI

QA_PROMPT = PromptTemplate(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context_str}\n\nQuestion: {query_str}\n\nAnswer:"
)
synth = get_response_synthesizer(
    llm=GoogleGenAI(
        model="gemini-3.6-flash",
        api_key=os.environ["GEMINI_API_KEY"],
        temperature=0.3,
    ),
    text_qa_template=QA_PROMPT,
)
response = synth.synthesize(question, nodes=hits)
print(response.response)

## 7. PLAYGROUND

Things to tweak and re-run cells 3–8:

- `sentences_per_chunk` (try 3 vs 10) — chunk granularity vs context size
- `similarity_top_k` (try 3 vs 20) — recall vs noise
- `hnsw_ef_search` (set `app_settings.hnsw_ef_search = 80` before cell 6) —
  HNSW search width vs recall/speed
- `sentence_overlap` (try 0 vs 2) — context continuity across chunk boundaries

Watch how the scores and the cited `p.N / section` change.